# Chapter 3: Cerulean City
## Observational Studies & Graphical Models

---

*"The water flows in many directions here in Cerulean City, and so do the paths of causation. Before you challenge the Gym, you must learn to read the currents."*

---

In Chapters 1 and 2 we worked with randomised experiments -- the gold standard for causal inference. But most data in the Pokemon world (and the real world) comes from **observation**, not experimentation. Trainers choose their own paths, pick their own strategies, and self-select into training regimes.

In this chapter we will learn:

1. How **confounding**, **selection bias**, and **Simpson's Paradox** can mislead us
2. How to use **Directed Acyclic Graphs (DAGs)** to map out causal structures
3. The three fundamental graph structures: **forks**, **chains**, and **colliders**
4. The **backdoor criterion** and **frontdoor criterion** for identifying causal effects

By the end, you will have earned the **Cascade Badge** and the tools to navigate observational data.

---
## Cell 1: Setup
---

In [ ]:
# ============================================================
# Cell 1 -- Setup: Load data, apply theme, set the scene
# ============================================================
import sys, pathlib

# Ensure the project root is on the path so kanto_utils is importable
PROJECT_ROOT = pathlib.Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import kanto_utils as ku

# Apply the Kanto visual theme
ku.apply_kanto_theme()

# Load datasets
trainers = ku.load_trainers()
battles  = ku.load_battles()

print(f"Trainers dataset : {trainers.shape[0]:,} trainers x {trainers.shape[1]} variables")
print(f"Battles  dataset : {battles.shape[0]:,} battles  x {battles.shape[1]} variables")
trainers.head(3)

In [ ]:
ku.oak_says(
    "Welcome to Cerulean City! In the real world -- and the Pokemon world -- "
    "we rarely get to run experiments. Instead, we <b>observe</b> trainers making "
    "their own choices. The challenge is that those choices are tangled up with "
    "other factors. Today, we learn to untangle them using <b>graphical models</b>."
)

---
## Cell 2: Confounding Demo

**Question:** Does training in Mt. Moon (cave_training) *cause* trainers to earn more badges?

---

In [ ]:
# ============================================================
# Cell 2 -- Confounding Demo
# ============================================================

# --- Naive comparison ---
naive = ku.difference_in_means(trainers["badges"], trainers["cave_training"])
print("=== Naive Comparison: cave_training -> badges ===")
print(f"  Mean badges (cave_training=1): {trainers.loc[trainers['cave_training']==1, 'badges'].mean():.3f}")
print(f"  Mean badges (cave_training=0): {trainers.loc[trainers['cave_training']==0, 'badges'].mean():.3f}")
print(f"  Naive estimate (biased):       {naive['estimate']:.3f}  (95% CI: [{naive['ci_lower']:.3f}, {naive['ci_upper']:.3f}])")
print()

# --- Show the confounder: trainer_experience ---
print("=== But trainer_experience confounds the relationship ===")
print(f"  Corr(trainer_experience, cave_training) = {trainers['trainer_experience'].corr(trainers['cave_training']):.3f}")
print(f"  Corr(trainer_experience, badges)         = {trainers['trainer_experience'].corr(trainers['badges']):.3f}")
print()

# --- Adjusted estimate: control for trainer_experience via OLS ---
from numpy.linalg import lstsq
X = np.column_stack([
    np.ones(len(trainers)),
    trainers["cave_training"].values,
    trainers["trainer_experience"].values,
])
y = trainers["badges"].values
beta, _, _, _ = lstsq(X, y, rcond=None)

print("=== OLS Adjusted for trainer_experience ===")
print(f"  cave_training coefficient (adjusted): {beta[1]:.3f}")
print(f"  trainer_experience coefficient:       {beta[2]:.3f}")
print(f"  The naive estimate ({naive['estimate']:.3f}) was BIASED UPWARD.")
print(f"  After adjustment, the effect shrinks to {beta[1]:.3f}.")
print()

In [ ]:
# Visualize the confounding
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Naive comparison
for label, val, color in [("No Cave", 0, "#3B4CCA"), ("Cave", 1, "#EE1515")]:
    sub = trainers[trainers["cave_training"] == val]
    axes[0].hist(sub["badges"], bins=range(0, 10), alpha=0.6, color=color,
                 label=f"{label} (mean={sub['badges'].mean():.2f})", edgecolor="white")
axes[0].set_xlabel("Badges earned")
axes[0].set_ylabel("Count")
axes[0].set_title("Naive: Cave Training vs Badges")
axes[0].legend(frameon=True)

# Panel 2: Confounder -- experience differs by group
for label, val, color in [("No Cave", 0, "#3B4CCA"), ("Cave", 1, "#EE1515")]:
    sub = trainers[trainers["cave_training"] == val]
    axes[1].hist(sub["trainer_experience"], bins=30, alpha=0.6, color=color,
                 label=f"{label} (mean={sub['trainer_experience'].mean():.2f})", edgecolor="white")
axes[1].set_xlabel("Trainer Experience")
axes[1].set_ylabel("Count")
axes[1].set_title("Confounder: Experience Differs by Group")
axes[1].legend(frameon=True)

# Panel 3: Scatter showing the full picture
for label, val, color in [("No Cave", 0, "#3B4CCA"), ("Cave", 1, "#EE1515")]:
    sub = trainers[trainers["cave_training"] == val]
    axes[2].scatter(sub["trainer_experience"], sub["badges"], alpha=0.25, s=15,
                    color=color, label=label)
axes[2].set_xlabel("Trainer Experience")
axes[2].set_ylabel("Badges")
axes[2].set_title("Badges vs Experience, Coloured by Cave Training")
axes[2].legend(frameon=True)

fig.tight_layout()
plt.show()

In [ ]:
ku.blues_mistake(
    claim="I trained in Mt. Moon and got more badges than you -- cave training obviously works!",
    reality=(
        "Trainers who choose cave training tend to have <b>more experience</b> to begin with. "
        "Experience causes both cave training (selection) and more badges (outcome). "
        "The naive comparison confounds the treatment effect with the experience advantage. "
        f"The biased estimate is {naive['estimate']:.3f}, but after adjusting for experience "
        f"the effect drops to {beta[1]:.3f}."
    )
)

---
## Cell 3: Selection Bias & Survivorship Bias

What happens when we only look at trainers who *made it* to Cerulean City?

---

In [ ]:
# ============================================================
# Cell 3 -- Selection Bias & Survivorship Bias
# ============================================================

# --- Survivorship bias: filter to trainers who reached Cerulean (badges >= 2) ---
cerulean_trainers = trainers[trainers["badges"] >= 2].copy()
print(f"Full dataset: {len(trainers)} trainers")
print(f"Reached Cerulean (badges >= 2): {len(cerulean_trainers)} trainers")
print(f"Dropped: {len(trainers) - len(cerulean_trainers)} trainers who never made it past Pewter\n")

# Effect in full sample
full_effect = ku.difference_in_means(trainers["badges"], trainers["cave_training"])
# Effect in survivorship-biased sample
surv_effect = ku.difference_in_means(cerulean_trainers["badges"], cerulean_trainers["cave_training"])

print(f"Cave training effect (FULL sample):      {full_effect['estimate']:.3f}")
print(f"Cave training effect (SURVIVORS only):    {surv_effect['estimate']:.3f}")
print(f"Conditioning on badges >= 2 changes the estimate!\n")

# --- Berkson's Paradox among Elite Four challengers ---
print("=== Berkson's Paradox: natural_talent vs play_hours ===")
elite_four = trainers[trainers["elite_four_attempted"] == 1].copy()
print(f"\nAll trainers:  corr(natural_talent, play_hours) = {trainers['natural_talent'].corr(trainers['play_hours']):.3f}")
print(f"Elite Four:    corr(natural_talent, play_hours) = {elite_four['natural_talent'].corr(elite_four['play_hours']):.3f}")
print(f"\nAmong ALL trainers, talent and hours are ~uncorrelated.")
print(f"But among Elite Four challengers, they appear NEGATIVELY correlated!")
print(f"This is Berkson's paradox: reaching the Elite Four is a collider.")

In [ ]:
# Visualize Berkson's Paradox
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: All trainers
axes[0].scatter(trainers["natural_talent"], trainers["play_hours"],
                alpha=0.15, s=10, color="#3B4CCA", label="All trainers")
r_all = trainers["natural_talent"].corr(trainers["play_hours"])
axes[0].set_xlabel("Natural Talent")
axes[0].set_ylabel("Play Hours")
axes[0].set_title(f"All Trainers (r = {r_all:.3f})")

# Panel 2: Elite Four challengers only
axes[1].scatter(trainers["natural_talent"], trainers["play_hours"],
                alpha=0.08, s=10, color="#CCCCCC", label="All trainers")
axes[1].scatter(elite_four["natural_talent"], elite_four["play_hours"],
                alpha=0.8, s=40, color="#EE1515", edgecolors="white",
                linewidths=0.5, label="Elite Four challengers", zorder=5)
# Fit line through Elite Four
m, b = np.polyfit(elite_four["natural_talent"], elite_four["play_hours"], 1)
x_line = np.linspace(elite_four["natural_talent"].min(), elite_four["natural_talent"].max(), 100)
axes[1].plot(x_line, m * x_line + b, "--", color="#EE1515", linewidth=2)
r_ef = elite_four["natural_talent"].corr(elite_four["play_hours"])
axes[1].set_xlabel("Natural Talent")
axes[1].set_ylabel("Play Hours")
axes[1].set_title(f"Elite Four Challengers Only (r = {r_ef:.3f})")
axes[1].legend(frameon=True)

fig.suptitle("Berkson's Paradox: Conditioning on a Collider Creates Spurious Correlation",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
ku.oak_says(
    "<b>Survivorship bias</b> occurs when we restrict our analysis to units that "
    "'survived' some process. Here, looking only at trainers who reached Cerulean "
    "excludes weak trainers without cave training, biasing our estimate.<br><br>"
    "<b>Berkson's paradox</b> is a form of collider bias. To reach the Elite Four, "
    "a trainer needs EITHER high talent OR lots of practice. So among Elite Four "
    "challengers, talent and practice <i>appear</i> negatively correlated -- "
    "but there is no such relationship in the full population."
)

---
## Cell 4: Simpson's Paradox

When the aggregate tells one story but every subgroup tells another.

---

In [ ]:
# ============================================================
# Cell 4 -- Simpson's Paradox
# ============================================================

# We'll demonstrate Simpson's Paradox using cave_training, starter_type, and badges.
# Key insight: the RATE of cave training varies by starter_type, and starter_type
# independently affects badges.

print("=" * 65)
print("SIMPSON'S PARADOX: cave_training x starter_type -> badges")
print("=" * 65)

# Build the summary table
rows = []
for starter in ["Fire", "Water", "Grass"]:
    sub = trainers[trainers["starter_type"] == starter]
    treated = sub[sub["cave_training"] == 1]
    control = sub[sub["cave_training"] == 0]
    rows.append({
        "Starter": starter,
        "N (Cave)": len(treated),
        "N (No Cave)": len(control),
        "Mean Badges (Cave)": treated["badges"].mean(),
        "Mean Badges (No Cave)": control["badges"].mean(),
        "Effect": treated["badges"].mean() - control["badges"].mean(),
        "Cave Rate": sub["cave_training"].mean(),
    })

# Aggregate
treated_all = trainers[trainers["cave_training"] == 1]
control_all = trainers[trainers["cave_training"] == 0]
rows.append({
    "Starter": "AGGREGATE",
    "N (Cave)": len(treated_all),
    "N (No Cave)": len(control_all),
    "Mean Badges (Cave)": treated_all["badges"].mean(),
    "Mean Badges (No Cave)": control_all["badges"].mean(),
    "Effect": treated_all["badges"].mean() - control_all["badges"].mean(),
    "Cave Rate": trainers["cave_training"].mean(),
})

simpson_table = pd.DataFrame(rows)
simpson_table = simpson_table.round(3)
print()
print(simpson_table.to_string(index=False))
print()

# Highlight the paradox
agg_effect = treated_all["badges"].mean() - control_all["badges"].mean()
within_effects = [r["Effect"] for r in rows if r["Starter"] != "AGGREGATE"]
avg_within = np.mean(within_effects)

print(f"Aggregate effect:             {agg_effect:.3f}")
print(f"Average within-group effect:  {avg_within:.3f}")
print(f"")
print("The aggregate effect and within-group effects tell the SAME direction here,")
print("but the MAGNITUDES differ. The aggregate is inflated by confounding from")
print("starter_type composition.")
print()
print("Let's construct a sharper example using synthetic modification...")

In [ ]:
# Construct a cleaner Simpson's Paradox illustration
# We'll create a synthetic variable to make the reversal crisp

np.random.seed(151)

# Simulate: Treatment = cave_training, Group = starter_type, Outcome = badge_improvement
# Design so that within each group, treatment has NEGATIVE effect,
# but aggregate has POSITIVE effect due to composition.

n_per = 200
sim_data = []

# Fire starters: high baseline badges, low cave training rate
for i in range(n_per):
    cave = np.random.binomial(1, 0.25)  # 25% do cave training
    badges = np.random.normal(6.0 - 0.3 * cave, 0.8)  # cave training has NEGATIVE effect
    sim_data.append({"starter": "Fire", "cave_training": cave, "badges": max(0, badges)})

# Water starters: medium baseline, medium cave rate
for i in range(n_per):
    cave = np.random.binomial(1, 0.50)
    badges = np.random.normal(4.0 - 0.3 * cave, 0.8)
    sim_data.append({"starter": "Water", "cave_training": cave, "badges": max(0, badges)})

# Grass starters: low baseline, high cave training rate
for i in range(n_per):
    cave = np.random.binomial(1, 0.75)  # 75% do cave training
    badges = np.random.normal(2.5 - 0.3 * cave, 0.8)
    sim_data.append({"starter": "Grass", "cave_training": cave, "badges": max(0, badges)})

sim_df = pd.DataFrame(sim_data)

# Show the paradox
print("=" * 65)
print("SIMPSON'S PARADOX (Constructed Example)")
print("=" * 65)
print()

for starter in ["Fire", "Water", "Grass"]:
    sub = sim_df[sim_df["starter"] == starter]
    t_mean = sub[sub["cave_training"]==1]["badges"].mean()
    c_mean = sub[sub["cave_training"]==0]["badges"].mean()
    t_rate = sub["cave_training"].mean()
    print(f"  {starter:6s} starters: Cave mean = {t_mean:.2f}, No Cave mean = {c_mean:.2f}, "
          f"Effect = {t_mean - c_mean:+.2f}  (cave rate = {t_rate:.0%})")

t_all = sim_df[sim_df["cave_training"]==1]["badges"].mean()
c_all = sim_df[sim_df["cave_training"]==0]["badges"].mean()
print(f"\n  {'AGGREGATE':6s}        : Cave mean = {t_all:.2f}, No Cave mean = {c_all:.2f}, "
      f"Effect = {t_all - c_all:+.2f}")

print(f"\n  Within EVERY starter group, cave training has a NEGATIVE effect.")
print(f"  But in the AGGREGATE, cave training appears to have a POSITIVE effect!")
print(f"  This is because Fire starters (high badges) rarely do cave training,")
print(f"  while Grass starters (low badges) frequently do.")

In [ ]:
# Visualize Simpson's Paradox
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: aggregate view (misleading)
for cave, color, label in [(0, "#3B4CCA", "No Cave"), (1, "#EE1515", "Cave")]:
    sub = sim_df[sim_df["cave_training"] == cave]
    axes[0].hist(sub["badges"], bins=20, alpha=0.6, color=color,
                 label=f"{label} (mean={sub['badges'].mean():.2f})", edgecolor="white")
axes[0].set_xlabel("Badges")
axes[0].set_title("Aggregate View: Cave Training Looks POSITIVE")
axes[0].legend(frameon=True)

# Right: within-group view (correct)
starter_colors = {"Fire": "#EE8130", "Water": "#6390F0", "Grass": "#7AC74C"}
offsets = {"Fire": -0.15, "Water": 0.0, "Grass": 0.15}
for starter in ["Fire", "Water", "Grass"]:
    sub = sim_df[sim_df["starter"] == starter]
    t_mean = sub[sub["cave_training"]==1]["badges"].mean()
    c_mean = sub[sub["cave_training"]==0]["badges"].mean()
    off = offsets[starter]
    axes[1].plot([0 + off, 1 + off], [c_mean, t_mean], "o-",
                 color=starter_colors[starter], linewidth=2, markersize=10,
                 label=f"{starter} ({t_mean - c_mean:+.2f})")
    axes[1].annotate(f"{c_mean:.2f}", (0 + off, c_mean), textcoords="offset points",
                     xytext=(-25, 5), fontsize=9, color=starter_colors[starter])
    axes[1].annotate(f"{t_mean:.2f}", (1 + off, t_mean), textcoords="offset points",
                     xytext=(5, 5), fontsize=9, color=starter_colors[starter])

axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["No Cave", "Cave"])
axes[1].set_ylabel("Mean Badges")
axes[1].set_title("Within-Group: Cave Training is NEGATIVE for All!")
axes[1].legend(frameon=True)

fig.suptitle("Simpson's Paradox", fontsize=15, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
ku.oak_says(
    "<b>Simpson's Paradox</b> occurs when a trend that appears in aggregated data "
    "reverses when the data is split into meaningful subgroups. The key question "
    "is: which analysis is correct? The answer depends on the <b>causal structure</b>. "
    "If starter_type is a confounder (affects both treatment and outcome), we should "
    "condition on it, and the within-group analysis is correct. To know which variables "
    "to condition on, we need <b>DAGs</b>."
)

---
## Cell 5: Building DAGs

Directed Acyclic Graphs -- the roadmap of causation.

---

In [ ]:
# ============================================================
# Cell 5 -- Building DAGs for the Kanto Trainer System
# ============================================================

# We'll draw the DAG using matplotlib for maximum compatibility.
# If you have graphviz installed, an alternative rendering is shown below.

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 7.5)
ax.set_aspect("equal")
ax.axis("off")

# Node positions
nodes = {
    "wealth":              (1.0, 6.5),
    "natural_talent":      (5.0, 7.0),
    "trainer_experience":  (1.0, 4.0),
    "starter_type":        (3.5, 6.0),
    "cave_training":       (2.5, 2.5),
    "strategy_score":      (5.5, 2.5),
    "play_hours":          (8.0, 5.5),
    "badges":              (8.5, 2.5),
    "elite_four":          (8.5, 0.5),
    "dedication":          (1.0, 1.0),
}

# Edges (cause -> effect)
edges = [
    ("wealth", "starter_type"),
    ("wealth", "trainer_experience"),
    ("natural_talent", "strategy_score"),
    ("natural_talent", "play_hours"),
    ("trainer_experience", "cave_training"),
    ("trainer_experience", "strategy_score"),
    ("trainer_experience", "badges"),
    ("cave_training", "strategy_score"),
    ("strategy_score", "badges"),
    ("play_hours", "badges"),
    ("starter_type", "badges"),
    ("dedication", "cave_training"),
    ("dedication", "badges"),
    ("badges", "elite_four"),
    ("play_hours", "elite_four"),
]

# Color coding
node_colors = {
    "cave_training": "#EE1515",   # Treatment
    "badges":        "#3B4CCA",   # Outcome
    "elite_four":    "#FFD733",   # Collider
}
default_color = "#7AC74C"

# Draw edges
for src, dst in edges:
    x0, y0 = nodes[src]
    x1, y1 = nodes[dst]
    ax.annotate("",
                xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(
                    arrowstyle="->",
                    color="#555555",
                    linewidth=1.5,
                    connectionstyle="arc3,rad=0.1",
                    shrinkA=18, shrinkB=18,
                ))

# Draw nodes
for name, (x, y) in nodes.items():
    color = node_colors.get(name, default_color)
    circle = plt.Circle((x, y), 0.4, facecolor=color, edgecolor="white",
                         linewidth=2.5, alpha=0.85, zorder=5)
    ax.add_patch(circle)
    # Label
    display_name = name.replace("_", "\n")
    ax.text(x, y, display_name, ha="center", va="center",
            fontsize=7, fontweight="bold", color="white", zorder=6)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#EE1515", edgecolor="white", label="Treatment (cave_training)"),
    Patch(facecolor="#3B4CCA", edgecolor="white", label="Outcome (badges)"),
    Patch(facecolor="#FFD733", edgecolor="white", label="Collider (elite_four)"),
    Patch(facecolor="#7AC74C", edgecolor="white", label="Other variables"),
]
ax.legend(handles=legend_elements, loc="lower left", frameon=True, fontsize=10)

ax.set_title("Kanto Trainer DAG: Causal Structure", fontsize=16, fontweight="bold", pad=15)
plt.show()

In [ ]:
ku.oak_says(
    "A <b>Directed Acyclic Graph (DAG)</b> encodes our assumptions about the "
    "causal data-generating process. Each arrow represents a direct causal effect. "
    "Key features of this DAG:<br><br>"
    "- <b>trainer_experience</b> is a confounder (common cause of cave_training and badges)<br>"
    "- <b>strategy_score</b> is a mediator (on the causal path from cave_training to badges)<br>"
    "- <b>elite_four</b> is a collider (common effect of badges and play_hours)<br>"
    "- <b>natural_talent</b> affects badges only through strategy_score (frontdoor opportunity)"
)

---
## Cell 6: Three-Structure Simulator

Every DAG is built from three fundamental structures: **Forks**, **Chains**, and **Colliders**.

---

In [ ]:
# ============================================================
# Cell 6 -- The Three Fundamental Structures
# ============================================================

np.random.seed(151)
n = 2000

# --- FORK: X <- C -> Y ---
# trainer_experience (C) causes both cave_training (X) and badges (Y)
C_fork = np.random.normal(5, 3, n)  # common cause
X_fork = 0.5 * C_fork + np.random.normal(0, 1, n)
Y_fork = 0.8 * C_fork + np.random.normal(0, 1, n)

# --- CHAIN: X -> M -> Y ---
# cave_training (X) -> strategy_score (M) -> badges (Y)
X_chain = np.random.normal(0, 1, n)
M_chain = 0.7 * X_chain + np.random.normal(0, 1, n)
Y_chain = 0.6 * M_chain + np.random.normal(0, 1, n)

# --- COLLIDER: X -> Col <- Y ---
# natural_talent (X) -> elite_four (Col) <- play_hours (Y)
X_coll = np.random.normal(0, 1, n)
Y_coll = np.random.normal(0, 1, n)
Col = 0.6 * X_coll + 0.6 * Y_coll + np.random.normal(0, 0.5, n)

# Compute correlations with and without conditioning
# For fork/chain: conditioning should REDUCE correlation
# For collider: conditioning should CREATE correlation

fig, axes = plt.subplots(3, 2, figsize=(14, 15))

structures = [
    ("FORK: X <- C -> Y", "trainer_exp causes both",
     X_fork, Y_fork, C_fork, "Condition on C (median split)"),
    ("CHAIN: X -> M -> Y", "cave -> strategy -> badges",
     X_chain, Y_chain, M_chain, "Condition on M (median split)"),
    ("COLLIDER: X -> Col <- Y", "talent -> elite_four <- hours",
     X_coll, Y_coll, Col, "Condition on Col (top quartile)"),
]

for row, (title, desc, X, Y, Z, cond_label) in enumerate(structures):
    # Unconditional
    r_uncond = np.corrcoef(X, Y)[0, 1]
    axes[row, 0].scatter(X, Y, alpha=0.15, s=10, color="#3B4CCA")
    m, b = np.polyfit(X, Y, 1)
    x_line = np.linspace(X.min(), X.max(), 100)
    axes[row, 0].plot(x_line, m * x_line + b, color="#EE1515", linewidth=2)
    axes[row, 0].set_title(f"{title}\nUnconditioned: r = {r_uncond:.3f}", fontsize=11)
    axes[row, 0].set_xlabel("X")
    axes[row, 0].set_ylabel("Y")

    # Conditional
    if row < 2:  # Fork and Chain: median split
        mask_lo = Z < np.median(Z)
        mask_hi = Z >= np.median(Z)
        for mask, color, label in [(mask_lo, "#3B4CCA", "Z < median"),
                                    (mask_hi, "#EE1515", "Z >= median")]:
            r_cond = np.corrcoef(X[mask], Y[mask])[0, 1]
            axes[row, 1].scatter(X[mask], Y[mask], alpha=0.15, s=10, color=color,
                                 label=f"{label} (r={r_cond:.3f})")
            m, b = np.polyfit(X[mask], Y[mask], 1)
            x_line = np.linspace(X[mask].min(), X[mask].max(), 100)
            axes[row, 1].plot(x_line, m * x_line + b, color=color, linewidth=2, linestyle="--")
    else:  # Collider: condition on top quartile
        mask_all = np.ones(n, dtype=bool)
        mask_coll = Col > np.percentile(Col, 75)
        r_cond = np.corrcoef(X[mask_coll], Y[mask_coll])[0, 1]
        axes[row, 1].scatter(X[~mask_coll], Y[~mask_coll], alpha=0.08, s=10,
                             color="#CCCCCC", label="Not conditioned")
        axes[row, 1].scatter(X[mask_coll], Y[mask_coll], alpha=0.4, s=15,
                             color="#EE1515", label=f"Col > 75th pctile (r={r_cond:.3f})")
        m, b = np.polyfit(X[mask_coll], Y[mask_coll], 1)
        x_line = np.linspace(X[mask_coll].min(), X[mask_coll].max(), 100)
        axes[row, 1].plot(x_line, m * x_line + b, color="#EE1515", linewidth=2, linestyle="--")

    axes[row, 1].set_title(f"{cond_label}", fontsize=11)
    axes[row, 1].set_xlabel("X")
    axes[row, 1].set_ylabel("Y")
    axes[row, 1].legend(frameon=True, fontsize=9)

fig.suptitle("The Three Fundamental DAG Structures", fontsize=15, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

print("\nSummary:")
print("  FORK:     X and Y are correlated. Conditioning on C BLOCKS the association.")
print("  CHAIN:    X and Y are correlated. Conditioning on M BLOCKS the association.")
print("  COLLIDER: X and Y are INDEPENDENT. Conditioning on Col OPENS a spurious path!")

In [ ]:
ku.gym_leader_says("Misty",
    "These three structures are the building blocks of ALL causal graphs. "
    "Master these, and you can read any DAG:<br><br>"
    "<b>Fork</b> (common cause): Condition on the common cause to block the non-causal path.<br>"
    "<b>Chain</b> (mediator): Do NOT condition on the mediator if you want the total effect.<br>"
    "<b>Collider</b> (common effect): Do NOT condition on it -- doing so CREATES bias!"
)

---
## Cell 7: Backdoor Criterion Checker

Given the DAG, which variables do we need to adjust for?

---

In [ ]:
# ============================================================
# Cell 7 -- Backdoor Criterion
# ============================================================

# Define the DAG as an adjacency list
dag = {
    "wealth":             ["starter_type", "trainer_experience"],
    "natural_talent":     ["strategy_score", "play_hours"],
    "trainer_experience": ["cave_training", "strategy_score", "badges"],
    "cave_training":      ["strategy_score"],
    "strategy_score":     ["badges"],
    "play_hours":         ["badges", "elite_four"],
    "starter_type":       ["badges"],
    "dedication":         ["cave_training", "badges"],
    "badges":             ["elite_four"],
    "elite_four":         [],
}

treatment = "cave_training"
outcome = "badges"

# Enumerate all undirected paths from treatment to outcome
def find_all_paths(graph, start, end):
    """Find all paths in the UNDIRECTED version of the DAG."""
    # Build undirected adjacency
    undirected = {}
    for node in graph:
        undirected.setdefault(node, set())
        for child in graph[node]:
            undirected.setdefault(child, set())
            undirected[node].add(child)
            undirected[child].add(node)
    
    paths = []
    stack = [(start, [start])]
    while stack:
        current, path = stack.pop()
        if current == end and len(path) > 1:
            paths.append(path)
            continue
        for neighbor in undirected.get(current, []):
            if neighbor not in path:
                stack.append((neighbor, path + [neighbor]))
    return paths

def get_path_edges(graph, path):
    """For each consecutive pair in path, determine if the edge goes forward or backward."""
    edges = []
    for i in range(len(path) - 1):
        a, b = path[i], path[i+1]
        if b in graph.get(a, []):
            edges.append((a, "->", b))
        else:
            edges.append((a, "<-", b))
    return edges

def is_backdoor_path(graph, path, treatment):
    """A backdoor path is one that starts with an arrow INTO the treatment."""
    if len(path) < 2:
        return False
    second = path[1]
    # Is there an arrow from second -> treatment? i.e., treatment in graph[second]
    return treatment in graph.get(second, [])

all_paths = find_all_paths(dag, treatment, outcome)

print(f"All paths from {treatment} to {outcome}:")
print(f"{'=' * 70}")
for path in sorted(all_paths, key=len):
    edges = get_path_edges(dag, path)
    edge_str = ""
    for a, direction, b in edges:
        if not edge_str:
            edge_str = a
        edge_str += f" {direction} {b}"
    is_bd = is_backdoor_path(dag, path, treatment)
    path_type = "BACKDOOR" if is_bd else "CAUSAL"
    print(f"  [{path_type:8s}] {edge_str}")

print()
print("=" * 70)
print("Backdoor Criterion Analysis:")
print("=" * 70)
print()
print("To identify the causal effect of cave_training on badges, we need to")
print("block all BACKDOOR paths without blocking CAUSAL paths.")
print()
print("Valid adjustment sets:")
print("  1. {trainer_experience, dedication}")
print("     - Blocks the backdoor through trainer_experience")
print("     - Blocks the backdoor through dedication")
print()
print("  2. {trainer_experience, dedication, wealth} (also valid, but wealth is redundant)")
print()
print("INVALID adjustment sets:")
print("  - {strategy_score}: This is a mediator! Controlling for it blocks the causal path.")
print("  - {elite_four}: This is a collider! Conditioning on it opens a non-causal path.")

In [ ]:
# Compute the adjusted estimates
print("=== Comparing Adjustment Strategies ===")
print()

# Naive (no adjustment)
naive_est = ku.difference_in_means(trainers["badges"], trainers["cave_training"])
print(f"  Naive (no adjustment):                            {naive_est['estimate']:.3f}")

# Adjust for trainer_experience + dedication (valid backdoor adjustment)
X_valid = np.column_stack([
    np.ones(len(trainers)),
    trainers["cave_training"].values,
    trainers["trainer_experience"].values,
    trainers["dedication"].values,
])
beta_valid, _, _, _ = np.linalg.lstsq(X_valid, trainers["badges"].values, rcond=None)
print(f"  Adjusted (experience + dedication):               {beta_valid[1]:.3f}  [VALID]")

# Adjust for strategy_score (BAD: blocks mediator)
X_bad = np.column_stack([
    np.ones(len(trainers)),
    trainers["cave_training"].values,
    trainers["strategy_score"].values,
])
beta_bad, _, _, _ = np.linalg.lstsq(X_bad, trainers["badges"].values, rcond=None)
print(f"  Adjusted (strategy_score -- MEDIATOR):            {beta_bad[1]:.3f}  [BIASED: blocks causal path!]")

# Adjust for everything including elite_four (BAD: opens collider)
X_coll = np.column_stack([
    np.ones(len(trainers)),
    trainers["cave_training"].values,
    trainers["trainer_experience"].values,
    trainers["elite_four_attempted"].values,
])
beta_coll, _, _, _ = np.linalg.lstsq(X_coll, trainers["badges"].values, rcond=None)
print(f"  Adjusted (experience + elite_four -- COLLIDER):   {beta_coll[1]:.3f}  [BIASED: opens collider path!]")
print()
print("Lesson: The adjustment set MATTERS. Only adjust for confounders, not mediators or colliders.")

In [ ]:
ku.oak_says(
    "The <b>Backdoor Criterion</b> (Pearl, 1995) tells us that a set of variables Z "
    "is sufficient for identifying the causal effect of X on Y if:<br><br>"
    "1. No node in Z is a descendant of X<br>"
    "2. Z blocks every path between X and Y that contains an arrow INTO X<br><br>"
    "When the backdoor criterion is satisfied, we can compute the causal effect by "
    "adjusting (conditioning) on Z. This is the foundation of observational causal inference."
)

---
## Cell 8: Frontdoor Criterion Demo

When we cannot observe the confounder, the frontdoor criterion offers another path to identification.

---

In [ ]:
# ============================================================
# Cell 8 -- Frontdoor Criterion
# ============================================================

# Scenario: We want to know the causal effect of cave_training on badges.
# natural_talent is an UNOBSERVED confounder (assume we can't measure it).
# But strategy_score is a MEDIATOR: cave_training -> strategy_score -> badges
# AND natural_talent does NOT directly affect cave_training (it goes through strategy_score).
#
# The frontdoor criterion says: if M (strategy_score) lies on the only causal
# path from X to Y, and no unobserved confounder affects both X and M, we can
# identify the causal effect via:
#   P(Y | do(X)) = sum_M P(M|X) * sum_X' P(Y|X',M) * P(X')

print("=" * 65)
print("FRONTDOOR CRITERION DEMO")
print("=" * 65)
print()
print("Scenario: natural_talent is UNOBSERVED.")
print("We cannot directly use the backdoor criterion.")
print()
print("But we observe strategy_score (mediator):")
print("  cave_training -> strategy_score -> badges")
print()
print("The frontdoor criterion allows us to identify the causal effect")
print("by decomposing it into two estimable pieces.")
print()

# Step 1: Effect of cave_training on strategy_score
# (unconfounded because natural_talent does not directly affect cave_training)
step1 = ku.difference_in_means(trainers["strategy_score"], trainers["cave_training"])
print(f"Step 1: E[strategy_score | cave=1] - E[strategy_score | cave=0] = {step1['estimate']:.3f}")

# Step 2: Effect of strategy_score on badges (adjusting for cave_training)
# This uses the backdoor criterion on the strategy_score -> badges subproblem
X_step2 = np.column_stack([
    np.ones(len(trainers)),
    trainers["strategy_score"].values,
    trainers["cave_training"].values,
])
beta_step2, _, _, _ = np.linalg.lstsq(X_step2, trainers["badges"].values, rcond=None)
print(f"Step 2: Coefficient of strategy_score on badges (adjusted for cave_training) = {beta_step2[1]:.4f}")

# Frontdoor estimate: product of the two steps
frontdoor_estimate = step1["estimate"] * beta_step2[1]
print(f"\nFrontdoor estimate: {step1['estimate']:.3f} x {beta_step2[1]:.4f} = {frontdoor_estimate:.3f}")
print()

# Compare with other estimates
print("=== Comparison of Estimates ===")
print(f"  Naive (biased):                {naive_est['estimate']:.3f}")
print(f"  Backdoor (if talent observed): {beta_valid[1]:.3f}")
print(f"  Frontdoor (talent unobserved): {frontdoor_estimate:.3f}")
print()
print("The frontdoor criterion recovers a causal estimate even though")
print("we cannot observe natural_talent!")

In [ ]:
# Visualize the frontdoor pathway
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Step 1: cave_training -> strategy_score
for val, color, label in [(0, "#3B4CCA", "No Cave"), (1, "#EE1515", "Cave")]:
    sub = trainers[trainers["cave_training"] == val]
    axes[0].hist(sub["strategy_score"], bins=30, alpha=0.6, color=color,
                 label=f"{label} (mean={sub['strategy_score'].mean():.1f})", edgecolor="white")
axes[0].set_xlabel("Strategy Score")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Step 1: cave_training -> strategy_score\n(effect = {step1['estimate']:.2f})")
axes[0].legend(frameon=True)

# Step 2: strategy_score -> badges
axes[1].scatter(trainers["strategy_score"], trainers["badges"], alpha=0.15, s=10, color="#3B4CCA")
x_line = np.linspace(trainers["strategy_score"].min(), trainers["strategy_score"].max(), 100)
axes[1].plot(x_line, beta_step2[0] + beta_step2[1] * x_line + beta_step2[2] * trainers["cave_training"].mean(),
             color="#EE1515", linewidth=2.5, label=f"slope = {beta_step2[1]:.4f}")
axes[1].set_xlabel("Strategy Score")
axes[1].set_ylabel("Badges")
axes[1].set_title(f"Step 2: strategy_score -> badges\n(adjusted for cave_training)")
axes[1].legend(frameon=True)

fig.suptitle(f"Frontdoor Criterion: {step1['estimate']:.2f} x {beta_step2[1]:.4f} = {frontdoor_estimate:.3f}",
             fontsize=14, fontweight="bold", y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
ku.oak_says(
    "The <b>Frontdoor Criterion</b> is a remarkable result. Even when we cannot "
    "observe a confounder, we can still identify a causal effect if we can observe "
    "a mediator M that satisfies:<br><br>"
    "1. X blocks all paths from any confounder to M<br>"
    "2. All causal paths from X to Y go through M<br>"
    "3. There is no unblocked backdoor path from M to Y (after conditioning on X)<br><br>"
    "Real-world applications are rare but powerful: think of the effect of "
    "tar on lung cancer, identified through smoking -> tar -> cancer."
)

---
## Challenges

Test your understanding of observational studies and graphical models.

---

### Challenge 1: Draw a New DAG

Consider the causal question: **Does Safari Zone access (`safari_zone_visits`) cause trainers to catch more Pokemon (`total_pokemon_caught`)?**

1. Draw a DAG for this question. Think about what confounders might exist (e.g., wealth, play_hours, patience).
2. Identify the valid adjustment set using the backdoor criterion.
3. Compute both the naive and adjusted estimates.

In [ ]:
# ============================================================
# Challenge 1: Your DAG and adjustment set
# ============================================================

# YOUR CODE HERE
# Suggested structure:
#   wealth -> safari_zone_visits, wealth -> play_hours
#   patience -> safari_zone_visits, patience -> total_pokemon_caught
#   play_hours -> total_pokemon_caught
#   safari_zone_visits -> total_pokemon_caught
#
# Step 1: Draw the DAG
# Step 2: Identify backdoor paths
# Step 3: Find the adjustment set

# Naive estimate
trainers["safari_any"] = (trainers["safari_zone_visits"] > 0).astype(int)
naive_safari = ku.difference_in_means(trainers["total_pokemon_caught"], trainers["safari_any"])
print(f"Naive effect of safari access on pokemon caught: {naive_safari['estimate']:.3f}")

# Adjusted estimate (adjust for confounders)
# Try adjusting for wealth and patience
X_adj = np.column_stack([
    np.ones(len(trainers)),
    trainers["safari_any"].values,
    trainers["wealth"].values,
    trainers["patience"].values,
])
beta_adj, _, _, _ = np.linalg.lstsq(X_adj, trainers["total_pokemon_caught"].values, rcond=None)
print(f"Adjusted effect (controlling for wealth, patience): {beta_adj[1]:.3f}")
print(f"\nDoes adjustment change the estimate? If so, the confounders matter!")

### Challenge 2: Create a Collider Bias Demonstration

Using the kanto_trainers data, find or construct a collider situation.

**Hint:** `total_battles_won` is caused by both `strategy_score` and `team_level_avg`. What happens if you condition on it?

In [ ]:
# ============================================================
# Challenge 2: Collider bias demonstration
# ============================================================

# YOUR CODE HERE
# 1. Check correlation between strategy_score and team_level_avg in full sample
r_full = trainers["strategy_score"].corr(trainers["team_level_avg"])
print(f"Full sample: corr(strategy_score, team_level_avg) = {r_full:.3f}")

# 2. Condition on total_battles_won (a collider) -- look at top quartile
high_wins = trainers[trainers["total_battles_won"] > trainers["total_battles_won"].quantile(0.75)]
r_cond = high_wins["strategy_score"].corr(high_wins["team_level_avg"])
print(f"Among high win trainers: corr(strategy_score, team_level_avg) = {r_cond:.3f}")

# 3. Visualize the two scenarios side by side
# (create a scatter plot showing the spurious correlation that appears)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(trainers["strategy_score"], trainers["team_level_avg"],
                alpha=0.15, s=10, color="#3B4CCA")
axes[0].set_title(f"Full Sample (r = {r_full:.3f})")
axes[0].set_xlabel("Strategy Score")
axes[0].set_ylabel("Team Level Avg")

axes[1].scatter(high_wins["strategy_score"], high_wins["team_level_avg"],
                alpha=0.3, s=15, color="#EE1515")
axes[1].set_title(f"Conditioned on High Wins (r = {r_cond:.3f})")
axes[1].set_xlabel("Strategy Score")
axes[1].set_ylabel("Team Level Avg")

fig.suptitle("Collider Bias: Conditioning on total_battles_won",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

### Challenge 3: Simpson's Paradox with a Different Variable

Can you find or construct another instance of Simpson's Paradox in the data?

**Hint:** Try grouping by `wealth` and looking at the effect of `exp_share_used` on `badges`.

In [ ]:
# ============================================================
# Challenge 3: Another Simpson's Paradox
# ============================================================

# YOUR CODE HERE
# Examine whether the effect of exp_share_used on badges
# reverses when you condition on wealth.

# Aggregate
treated_exp = trainers[trainers["exp_share_used"] == 1]
control_exp = trainers[trainers["exp_share_used"] == 0]
agg_effect = treated_exp["badges"].mean() - control_exp["badges"].mean()
print(f"Aggregate effect of exp_share on badges: {agg_effect:.3f}")
print()

# Within wealth groups
for w in sorted(trainers["wealth"].unique()):
    sub = trainers[trainers["wealth"] == w]
    t = sub[sub["exp_share_used"] == 1]["badges"]
    c = sub[sub["exp_share_used"] == 0]["badges"]
    if len(t) > 5 and len(c) > 5:
        eff = t.mean() - c.mean()
        print(f"  Wealth={w}: effect = {eff:.3f}  (n_treated={len(t)}, n_control={len(c)})")

print("\nDoes the direction change within groups vs aggregate?")

### Challenge 4: Implement a d-Separation Checker

Write a function that takes a DAG (as an adjacency list), two nodes, and a conditioning set, and returns whether the two nodes are d-separated given the conditioning set.

In [ ]:
# ============================================================
# Challenge 4: d-Separation Checker
# ============================================================

def d_separated(dag, node_a, node_b, conditioned_on):
    """
    Check if node_a and node_b are d-separated given conditioned_on in the DAG.
    
    Uses the Bayes-Ball algorithm:
    - Start from node_a
    - Try to reach node_b by traversing the graph
    - Respect the d-separation rules for forks, chains, and colliders
    
    Parameters
    ----------
    dag : dict
        Adjacency list {parent: [children, ...]}
    node_a : str
        Start node
    node_b : str  
        Target node
    conditioned_on : set
        Set of nodes being conditioned on
    
    Returns
    -------
    bool
        True if node_a and node_b are d-separated given conditioned_on
    """
    # YOUR CODE HERE
    # Hint: Build a parent lookup, then implement the Bayes-Ball algorithm.
    # The algorithm uses a queue of (node, direction) tuples where direction
    # indicates whether we arrived at the node from a parent ("down") or
    # child ("up").
    
    conditioned_on = set(conditioned_on)
    
    # Build parent lookup
    parents = {node: [] for node in dag}
    for parent, children in dag.items():
        for child in children:
            if child not in parents:
                parents[child] = []
            parents[child].append(parent)
    
    # Also ensure all nodes exist in dag
    children_of = dict(dag)
    for node in parents:
        if node not in children_of:
            children_of[node] = []
    
    # Get all descendants of conditioned nodes (needed for collider check)
    def get_ancestors(node_set):
        """Get all ancestors of nodes in node_set."""
        ancestors = set()
        queue = list(node_set)
        while queue:
            n = queue.pop()
            for p in parents.get(n, []):
                if p not in ancestors:
                    ancestors.add(p)
                    queue.append(p)
        return ancestors
    
    ancestors_of_conditioned = get_ancestors(conditioned_on)
    
    # Bayes-Ball: BFS with (node, arrived_via_parent) states
    # arrived_via_parent = True means we came DOWN from a parent
    # arrived_via_parent = False means we came UP from a child
    visited = set()
    reachable = set()
    queue = []
    
    # Start: try both directions from node_a
    queue.append((node_a, True))   # as if arrived from parent
    queue.append((node_a, False))  # as if arrived from child
    
    while queue:
        current, from_parent = queue.pop(0)
        
        if (current, from_parent) in visited:
            continue
        visited.add((current, from_parent))
        reachable.add(current)
        
        if current not in conditioned_on and from_parent:
            # Arrived from parent, node not conditioned: can pass to children
            for child in children_of.get(current, []):
                queue.append((child, True))
            # Can also pass to other parents (fork structure)
            for parent in parents.get(current, []):
                queue.append((parent, False))
        
        elif current not in conditioned_on and not from_parent:
            # Arrived from child, node not conditioned: can pass to parents only
            for parent in parents.get(current, []):
                queue.append((parent, False))
        
        elif current in conditioned_on and from_parent:
            # Arrived from parent, node IS conditioned: blocked (chain/fork)
            pass
        
        elif current in conditioned_on and not from_parent:
            # Arrived from child, node IS conditioned: can pass to parents AND children
            # (This is the collider unblocking rule)
            for parent in parents.get(current, []):
                queue.append((parent, False))
            for child in children_of.get(current, []):
                queue.append((child, True))
        
        # Also check: if current is an ancestor of a conditioned node and
        # we arrived from a child, the collider path is open
        if not from_parent and current in ancestors_of_conditioned and current not in conditioned_on:
            for parent in parents.get(current, []):
                queue.append((parent, False))
    
    return node_b not in reachable


# Test the d-separation checker
print("=== Testing d-Separation Checker ===")
print()

tests = [
    ("cave_training", "badges", set(), False,
     "cave_training and badges: connected (direct + confounded)"),
    ("cave_training", "badges", {"trainer_experience", "dedication", "strategy_score", "play_hours", "starter_type"}, True,
     "cave_training and badges | {experience, dedication, strategy, hours, starter}: d-separated"),
    ("natural_talent", "play_hours", set(), False,
     "natural_talent and play_hours: connected (direct cause)"),
    ("natural_talent", "cave_training", set(), True,
     "natural_talent and cave_training: d-separated (no connecting path)"),
]

for a, b, cond, expected, desc in tests:
    result = d_separated(dag, a, b, cond)
    status = "PASS" if result == expected else "FAIL"
    print(f"  [{status}] {desc}")
    print(f"         d-separated = {result} (expected {expected})")

---

## Chapter Summary

In Cerulean City we learned:

- **Confounding** arises when a common cause affects both treatment and outcome
- **Selection bias** and **survivorship bias** distort estimates when we condition on post-treatment variables
- **Simpson's Paradox** shows that aggregate and subgroup effects can point in opposite directions
- **DAGs** provide a visual language for encoding causal assumptions
- The **three fundamental structures** (fork, chain, collider) govern how information flows
- The **backdoor criterion** identifies valid adjustment sets
- The **frontdoor criterion** enables identification even with unobserved confounders

---

In [ ]:
ku.badge_earned("Cascade", 3)